# CS383: Data Science and Machine Learning
## Lecture 7 — Linear Regression

*Dr. Thitima Srivatanakul*

### Guiding question
**How does a model turn a scatter of points into a single "best" line — and how do you tell the
difference between a model that found something real and one that's honestly telling you it found
nothing?**

### Learning objectives
By the end of this lecture, you should be able to:

- explain what regression predicts (a numeric target), identify $X$, $y$, $\hat{y}$, the intercept, and
  the slope in a fitted model, and explain how least squares finds a best-fitting line (without needing
  to derive the formula yourself);
- extend that idea to multiple features, knowing when a feature needs encoding first;
- build a train/test + preprocessing workflow (with `ColumnTransformer` or `Pipeline`) that never lets
  test-set information leak into training;
- compute and interpret MAE, RMSE, and R² — including why a higher R² doesn't automatically mean a
  better or valid model — and use ablation (fitting a feature alone vs. combined) to check which feature
  is actually driving a model's R²;
- read a residual plot for basic checks, and recognize **target leakage**: a feature that's derived from
  the very outcome it's predicting.

---

### Where this fits
Lecture 6 already prepped two datasets for exactly this moment: NYC 311's `resolution_time_hours` and
restaurant inspection features (encoded, scaled). Today we build a regression model from scratch — NYC
311 first, restaurant inspections second. The second dataset comes with a twist worth watching for.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. Part 1 builds intuition for what a regression line actually is — eyeballing a
trend, picking the best of several candidate lines, and seeing a real dataset where the honest answer is
"barely any trend at all." Part 2 works out the math behind "best fit," visually: what a residual is, why
squaring it matters, and a hands-on hunt for the least-squares line before the formula is revealed.

**[Open the "What Is Regression, Really?" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect07/regression_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Simple Linear Regression, From Scratch

**Regression predicts a numeric target.** Before letting scikit-learn do it for us, let's see exactly
what "fitting a line" means, using one feature at a time.

### Setup — NYC 311 complaints

Same source Lecture 6 flagged as "a real regression target in Week 7": `resolution_time_hours` (how long
a complaint took to close), alongside `complaint_type`, `borough`, and the hour it was filed. We also
drop the handful of complaints logged under a non-standard `"Unspecified"` borough (too few to fit a
reliable coefficient for), and bucket every complaint type outside the 15 most common ones into
`"Other"` — with 100+ distinct complaint types in the raw data, some appearing only once, a full one-hot
encoding would otherwise let a single row masquerade as a pattern.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room. Deliberately built with
    # resolution time unrelated to complaint_type -- see Part 3 for why the real snapshot behaves
    # differently.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

# Keep only complaints that actually closed (resolution_time_hours is defined), and only the 5
# standard NYC boroughs -- a handful of rows are tagged "Unspecified," too rare to fit a reliable
# coefficient for.
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)

# complaint_type has 100+ distinct values in the real data, many with only a handful of rows -- group
# everything outside the 15 most common types into "Other" so a one-hot encoding doesn't end up
# treating a single rare row as if it were a reliable pattern.
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "complaint_grouped", "borough", "hour_filed", "resolution_time_hours"]].head()

In [ ]:
plt.scatter(complaints_df["hour_filed"], complaints_df["resolution_time_hours"], alpha=0.15, s=10)
plt.ylim(0, 300)  # zoomed in purely for visibility -- more on the points above this line later
plt.xlabel("Hour complaint was filed (0-23)")
plt.ylabel("Resolution time (hours)")
plt.title("Resolution Time vs. Hour Filed")
plt.show()

Across the full 0-23 hour range, the cloud looks about the same height everywhere — no obvious upward or
downward drift as the hour changes. (The y-axis above is zoomed to the first 300 hours purely for
visibility; a small number of complaints take far longer than that to resolve.)

### The pieces of a regression model

- **$X$** (feature / input): here, `hour_filed` — the hour a complaint was filed.
- **$y$** (actual target): `resolution_time_hours` — how long the complaint actually took to close.
- **$\hat{y}$** ("y-hat," predicted target): the model's *guess* at resolution time, not the real one.

A simple linear regression model connects them with a straight line:

$$\hat{y} = b_0 + b_1 x$$

- **$b_0$ (intercept)**: the predicted resolution time when `hour_filed` is exactly 0 (midnight).
- **$b_1$ (slope / coefficient)**: how much the predicted resolution time changes for each additional
  hour later in the day a complaint is filed.
- **residual**: how far off a single prediction was, $\text{residual} = y - \hat{y}$. A residual of 0 is
  a perfect prediction for that row; a large residual (positive or negative) means the model missed by a
  lot.

Every choice of $b_0$ and $b_1$ draws a different line. "Fitting" the model means picking the *one* line
that fits this data best — regardless of whether that best line turns out to be a good line.

### Finding the "best" line: least squares

A good line has small residuals across the board. To turn "small residuals across the board" into one
number we can minimize, linear regression adds up the *squared* residuals:

$$RSS = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**RSS** stands for **residual sum of squares**. Why square them instead of just adding residuals
directly? Two reasons: squaring makes every term positive, so overestimates and underestimates can't
cancel out — and it punishes big misses more than small ones (a residual of 10 counts 100x as much as a
residual of 1).

**Linear regression chooses the $b_0$ and $b_1$ that make RSS as small as possible.** That one idea is
what everything else in this lecture builds on.

### There's a direct formula for the best line

For simple regression (one feature), there's a direct formula for the $b_0$ and $b_1$ that minimize RSS
— you don't need to derive it, just recognize what it's doing:

$$b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad b_0 = \bar{y} - b_1 \bar{x}$$

In words: the slope compares how $x$ and $y$ move together (numerator) to how much $x$ varies on its own
(denominator). The intercept just makes the line pass through $(\bar{x}, \bar{y})$, the "center" of the
data. Notice this formula has no opinion about whether $x$ and $y$ are actually related — if they don't
move together at all, it will honestly hand back a slope near zero.

In [ ]:
x = complaints_df["hour_filed"].values
y = complaints_df["resolution_time_hours"].values

x_bar = x.__________()
y_bar = y.__________()

b1_byhand = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
b0_byhand = y_bar - b1_byhand * x_bar

print(f"By-hand intercept (b0): {b0_byhand:.4f}")
print(f"By-hand slope     (b1): {b1_byhand:.4f}")

In [ ]:
simple_model = __________()
simple_model.fit(complaints_df[["hour_filed"]], complaints_df["resolution_time_hours"])

print(f"scikit-learn intercept: {simple_model.intercept_:.4f}")
print(f"scikit-learn slope:     {simple_model.coef_[0]:.4f}")

Same numbers (up to rounding). `LinearRegression()` isn't doing anything mysterious — it's solving the
exact same minimization problem we just did by hand, and it's just as honest about a near-zero slope as
our by-hand version was.

In [ ]:
x_range = np.__________(0, 23, 100)
y_line = b0_byhand + b1_byhand * x_range

plt.scatter(complaints_df["hour_filed"], y, alpha=0.15, s=10, label="Actual complaints")
plt.plot(x_range, y_line, color="#D98C3F", linewidth=2, label="Fitted line")
plt.ylim(0, 300)
plt.xlabel("Hour complaint was filed (0-23)")
plt.ylabel("Resolution time (hours)")
plt.title("The Least-Squares Line")
plt.legend()
plt.show()

The fitted line is nearly flat — about half an hour of extra resolution time per hour later in the day,
essentially nothing next to typical resolution times. That's not a plotting mistake — it's the honest
least-squares answer for this particular $x$ and $y$. A flat line is still a real fitted model; it's just
a model whose best prediction barely changes no matter what you feed it.

*Note:* we're treating `hour_filed` as an ordinary number here, which is fine for this introductory
example. But time of day is actually cyclical — hour 23 and hour 0 are one hour apart, not 23. A model
that needs to capture that wraparound would need a different encoding (e.g., sine/cosine features); we
won't need that here.

---

## Part 2 — Multiple Linear Regression

Same idea as simple regression, but now $X$ contains several features instead of just one:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_p x_p$$

Each $b$ is still a coefficient, one per feature, and the model is still found by minimizing RSS — just
over more coefficients at once. Let's add `borough` alongside `hour_filed`.

### A quick reminder: `borough` needs encoding first

`borough` is categorical (text labels like `"MANHATTAN"`, not numbers), and a regression model can only
work with numbers. You already know the fix from Lecture 6: one-hot encode it with `OneHotEncoder`,
typically inside a `ColumnTransformer` alongside any numeric features that need scaling. We'll use exactly
that pattern in Part 3 — no need to relearn it here.

### *(Optional background)* The matrix version

Curious how scikit-learn actually solves for many coefficients at once? Stack a column of 1s onto your
feature matrix (for the intercept), and the whole model becomes $\hat{y} = X\beta$, where $\beta$ is the
vector of all the $b$'s. Minimizing RSS then has a closed-form solution, the **normal equation**:

$$\beta = (X^T X)^{-1} X^T y$$

You will not need to compute this by hand in this course — `LinearRegression` handles it internally
(with a more numerically stable method than a literal matrix inversion). It's here so the idea "many
coefficients, one formula" has a name if you see it again.

---

## Part 3 — Evaluating a Regression Model, the Right Way

Time to see multiple regression in action, and to evaluate it properly: split first, fit preprocessing
on the training data only, then fit the model — never let information from the test set leak into
training. We'll build the full model this time: `hour_filed` + `borough` + `complaint_grouped`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    complaints_df[["hour_filed", "borough", "complaint_grouped"]],
    complaints_df["resolution_time_hours"],
    test_size=__________, random_state=383,
)

preprocessor_311 = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["borough", "complaint_grouped"]),
])
X_train_ready = preprocessor_311.fit_transform(X_train)
X_test_ready = preprocessor_311.transform(X_test)

eval_model = LinearRegression()
eval_model.fit(X_train_ready, y_train)
y_pred = eval_model.predict(X_test_ready)

print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))

Six steps, in this order: **split** the data, **fit** the preprocessor on the training data only,
**transform** the training data, **transform** the test data using those *same* training-learned rules,
**fit** the model on the transformed training data, **predict** on the transformed test data. Fitting the
preprocessor on the test set (or on the full dataset before splitting) would let information about the
test set leak into training — an easy mistake that quietly makes your evaluation too optimistic.

### A shortcut: bundling it into a `Pipeline`

Calling `.fit_transform()`, then `.transform()`, then `.fit()`, then `.predict()` by hand works, but
scikit-learn's `Pipeline` bundles preprocessing and the model into a single object, so the steps can't
accidentally happen out of order:

In [ ]:
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("preprocessor", preprocessor_311),
    ("regressor", LinearRegression()),
])

model.__________(X_train, y_train)
y_pred_pipeline = model.predict(X_test)

print("Same R² as the manual version above?", np.isclose(r2_score(y_test, y_pred), r2_score(y_test, y_pred_pipeline)))

Same result, packaged more safely:

```
fit():
    preprocessing learns from training data
    -> training data is transformed
    -> regression model learns

predict():
    test data is transformed using the same rules learned above
    -> model makes predictions
```

`Pipeline.fit()` calls the preprocessor's `fit_transform()` on the training data, then fits the regressor
on the result. `Pipeline.predict()` calls the preprocessor's `transform()` — never `fit_transform()`
again — on new data, then predicts. One object, which greatly reduces the risk of accidentally re-fitting
the preprocessor on test data. The rest of this lecture keeps using the manual `eval_model` / `y_pred`
from above; the `Pipeline` is here so you recognize the pattern.

### Mean Absolute Error (MAE)

$$MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

The average size of a miss, in the original units (hours, here) — the most directly interpretable
metric.

In [ ]:
mae = __________(y_test, y_pred)
print(f"MAE: {mae:.2f} hours")

### RMSE (and its ingredient, MSE)

$$MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2 \qquad RMSE = \sqrt{MSE}$$

MSE squares the errors first (same reasoning as RSS), which makes large misses count disproportionately
more. RMSE takes the square root at the end, bringing the units back to the original scale (hours) so
it's directly comparable to MAE — and $RMSE \geq MAE$ always. A big gap between them means a few large
errors are dragging the average up; a small gap means errors are fairly uniform in size.

In [ ]:
mse = __________(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f} hours")

### R² (coefficient of determination)

$$R^2 = 1 - \frac{RSS}{\sum (y_i - \bar{y})^2}$$

The denominator is how much error you'd make with the simplest possible model: always predicting the
mean, $\bar{y}$. R² is the fraction of that baseline error your model *eliminates*.

- $R^2 = 1$: perfect predictions.
- $R^2 = 0$: your model does no better than just guessing the mean every time.
- $R^2 < 0$: your model does *worse* than guessing the mean — a real possibility, not just a theoretical
  edge case.

**Important: R² = 0.80 does not mean "80% accuracy."** It means the model explains 80% of the variation
in the target, compared to always guessing the mean — a different statement, not a percentage of correct
predictions.

In [ ]:
rss = np.sum((y_test - y_pred) ** 2)
tss = np.sum((y_test - y_test.mean()) ** 2)
r2_byhand = 1 - rss / tss

r2 = __________(y_test, y_pred)

print(f"By-hand R²: {r2_byhand:.4f}")
print(f"sklearn R²: {r2:.4f}")

### Which feature is actually doing the work?

R² tells you how much of the variation in `resolution_time_hours` this three-feature model explains
overall. It doesn't tell you how much each feature is contributing on its own. Let's check, by fitting
progressively larger models on the same train/test split and comparing R² at each step.

In [ ]:
model_hour_only = LinearRegression().fit(X_train[["hour_filed"]], y_train)
r2_hour_only = r2_score(y_test, model_hour_only.predict(X_test[["hour_filed"]]))

preprocessor_hb = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["borough"]),
])
X_train_hb = preprocessor_hb.fit_transform(X_train[["hour_filed", "borough"]])
X_test_hb = preprocessor_hb.transform(X_test[["hour_filed", "borough"]])
model_hour_borough = LinearRegression().fit(X_train_hb, y_train)
r2_hour_borough = __________(y_test, model_hour_borough.predict(X_test_hb))

print(f"hour_filed alone:                     R² = {r2_hour_only:.4f}")
print(f"hour_filed + borough:                 R² = {r2_hour_borough:.4f}")
print(f"hour_filed + borough + complaint_type: R² = {r2:.4f}")

In [ ]:
complaints_df.groupby("complaint_grouped")["resolution_time_hours"].__________().sort_values()

`hour_filed` alone and `hour_filed` + `borough` both land at essentially R² = 0. But adding
`complaint_grouped` jumps R² to about 0.17 — a real, meaningful chunk of the variation in resolution
time, not a rounding artifact.

Is this leakage, the way it might look at first glance? No. `complaint_type` is known the moment a
complaint is filed, before any resolution happens — it isn't computed from `resolution_time_hours` the
way a feature in Part 5 turns out to be computed from its target. The median-by-type table above makes
the pattern obvious: quick-response types like `Noise - Commercial` or `Illegal Fireworks` typically
close in about an hour, while infrastructure complaints like `UNSANITARY CONDITION`, `PLUMBING`, or
`HEAT/HOT WATER` typically take days to months. Different complaint types genuinely do get handled on
different timelines — genuine, non-circular signal.

The habit matters more than this specific result: whenever adding a feature causes a disproportionate
jump in R², ask *why* before trusting the higher number. Here, the answer holds up. In Part 5, it won't.

---

## Part 4 — Checking the Model: Residual Plots

A single R² number can hide a lot. A residual plot is a quick visual check — look for three things:

1. Are the residuals roughly **centered around 0**?
2. Is there an obvious **curve** (a sign the true relationship isn't a straight line)?
3. Is there an obvious **funnel** shape (errors growing or shrinking as predictions get bigger)?

In [ ]:
residuals = y_test - __________

plt.scatter(y_pred, residuals, alpha=0.3, s=12)
plt.axhline(0, color="#9AA5B1", linestyle="--")
plt.xlabel("Predicted resolution time (hours)")
plt.ylabel("Residual")
plt.title("Residuals vs. Predicted Resolution Time")
plt.show()

No obvious funnel or curve here — mostly a wide, roughly centered scatter, with a handful of points
running much higher than the rest (that long tail comes from the target's skew — see the optional section
below). This plot is a decent check in a narrow sense: the model isn't missing an obvious pattern it
should have caught. But "a clean residual plot" and "this model explains a lot" are different claims —
R² answers the second one, not this plot.

In [ ]:
plt.__________(residuals, bins=30, color="#2E5C8A", edgecolor="white")
plt.xlabel("Residual")
plt.title("Distribution of Residuals")
plt.show()

Roughly centered on 0, which is good — but not symmetric: a tall spike of small residuals, and a long
thin tail stretching toward large positive values. That shape comes directly from the target itself,
which is heavily right-skewed (more in the optional section below).

---

### *Optional / Advanced:* right-skewed targets and the log-transform

Not a required concept for this course — feel free to skip ahead to Part 5. It's here for anyone curious
why the residual plots above looked the way they did, and as a technique you may find useful later.

In [ ]:
plt.hist(y_train, bins=40, color="#2E5C8A", edgecolor="white")
plt.xlabel("Resolution time (hours)")
plt.title("Distribution of Resolution Time (Training Set)")
plt.show()

print(f"Mean:   {y_train.mean():.1f} hours")
print(f"Median: {y_train.median():.1f} hours")

A mean of about 78 hours next to a median of about 2 hours is a clear right-skew signature: about half of
all complaints in the training set close within roughly 2 hours, while a long tail of slow ones (some
taking weeks or months) pulls the mean upward, to nearly 35x the median. A method that minimizes
*squared* error lets those rare extreme cases pull the fitted line around disproportionately — the same
long tail of unusually large residuals we saw in the residual plot above.

### A common fix: log-transform the target

Applying `log1p()` (log of $1+x$, so it stays defined at 0) compresses that long right tail, making the
target's distribution look a lot closer to symmetric — often a better match for what a straight line can
actually capture. "Often" is doing real work in that sentence, though — let's actually check.

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

model_log = LinearRegression()
model_log.fit(X_train_ready, y_train_log)
y_pred_log = model_log.predict(X_test_ready)

r2_log = r2_score(y_test_log, y_pred_log)
print(f"R² on raw hours:              {r2:.4f}")
print(f"R² on log-transformed target: {r2_log:.4f}")

That's a big jump — R² roughly triples, from about 0.17 on the raw scale to about 0.57 on the
log-transformed scale. Some caution is still warranted: these two R² values are measured on different
scales (hours vs. log-hours), so a higher R² on log-hours doesn't automatically mean better predictions
once you convert back to real hours. But a jump this large, on a target this skewed, is a real signal
that the transform is helping the model fit the data it's actually built to fit. If you need predictions
back in real hours, apply `np.expm1()` to reverse the transform, and re-check MAE/RMSE on that original
scale before claiming victory.

---

## Part 5 — A Second, Trickier Case: Restaurant Inspection Scores

311 just showed a big jump in R² that turned out to be genuine. Now here's a second dataset — NYC
restaurant inspections — set up as a deliberate teaching example of target leakage, not a model anyone
would actually deploy. A similarly big jump in R² shows up here, for a very different reason. Same
question every time: *why* did that happen?

### Setup — NYC restaurant inspections

Same source as Lecture 6, rebuilt at the row level (one row per violation citation) rather than
collapsed into groups — we want `is_critical` and `grade` to describe an actual real record, not an
artifact of how rows happened to get grouped.

In [ ]:
try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "grade"]).reset_index(drop=True)

    # One row here is one violation citation, not one full inspection -- be careful not to call this
    # "violations per inspection," since a single inspection can contribute more than one row.
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)
    inspections_df["grade_ord"] = inspections_df["grade"].map({"A": 0, "B": 1, "C": 2})

    # grade also includes non-letter administrative statuses (N = not yet graded, Z = grade
    # pending, P = grade pending issued on reopening) that have no ordinal meaning -- drop those
    # so grade_ord never contains NaN wherever it's used as a model feature below.
    inspections_df = inspections_df.dropna(subset=["grade_ord"]).reset_index(drop=True)
    live_restaurants = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room. Built so grade is
    # derived from score (the same way NYC actually assigns letter grades), and is_critical is only
    # weakly related to score -- so the Part 5 leakage lesson still holds up even offline.
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    score = rng.integers(0, 71, size=n)
    grade = np.where(score <= 13, "A", np.where(score <= 27, "B", "C"))

    # A critical violation nudges the score up a little, but most of the variation in score has
    # nothing to do with it -- a weak, noisy relationship, unlike grade's near-exact one.
    is_critical = (rng.random(n) < (0.25 + score / 300)).astype(int)

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "grade": grade,
        "critical_flag": np.where(is_critical == 1, "Critical", "Not Critical"),
        "is_critical": is_critical,
    })
    inspections_df["grade_ord"] = inspections_df["grade"].map({"A": 0, "B": 1, "C": 2})
    live_restaurants = False

print(f"{'Shared snapshot' if live_restaurants else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_description", "score", "grade", "is_critical"]].head()

### Fitting a two-feature model

`is_critical` and `grade_ord` are both already numeric (no encoding needed — `grade_ord` is Lecture 6's
ordinal encoding of `grade`). Split, fit, predict — same workflow as Part 3.

In [ ]:
X_scores = inspections_df[["is_critical", "grade_ord"]]
y_scores = inspections_df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scores, y_scores, test_size=__________, random_state=383
)

eval_model = LinearRegression()
eval_model.fit(X_train, y_train)
y_pred = eval_model.__________(X_test)

print("Coefficients [is_critical, grade_ord]:", eval_model.coef_.round(4))
print("Intercept:", round(eval_model.intercept_, 4))

### Interpreting the coefficients

`is_critical`'s coefficient is under 1 point. `grade_ord`'s is around 15 points *per grade step* — over
an order of magnitude larger, on a feature that only ever takes the values 0, 1, or 2.

That gap is worth stopping on, not skimming past. Whenever a predictor's effect looks that outsized, ask:
*could I have known this feature before the target outcome, and was it created independently of that
outcome?* Let's evaluate the model properly first, then come back to that question.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = __________(y_test, y_pred)

print(f"MAE:  {mae:.2f} points")
print(f"RMSE: {rmse:.2f} points")
print(f"R²:   {r2:.4f}")

### Which feature is actually doing the work?

Same check as Part 3 — fit three separate models on the same train/test split (`is_critical` alone,
`grade_ord` alone, and both together) and compare their R² side by side.

In [ ]:
model_critical_only = LinearRegression().fit(X_train[["is_critical"]], y_train)
r2_critical_only = r2_score(y_test, model_critical_only.predict(X_test[["is_critical"]]))

model_grade_only = LinearRegression().fit(X_train[["grade_ord"]], y_train)
r2_grade_only = __________(y_test, model_grade_only.predict(X_test[["grade_ord"]]))

print(f"is_critical alone:       R² = {r2_critical_only:.4f}")
print(f"grade_ord alone:         R² = {r2_grade_only:.4f}")
print(f"is_critical + grade_ord: R² = {r2:.4f}")

In [ ]:
inspections_df.groupby("grade")["score"].__________()

`is_critical` alone gets an R² of about 0.02 — genuinely near zero. But `grade_ord` alone gets an R² of
about 0.76, and adding `is_critical` barely moves that number. Almost all of the combined model's power
was coming from `grade_ord` alone.

Ask the question from a moment ago: *could I have known `grade_ord` before the score existed, and was it
created independently of the score?* No. NYC assigns letter grades by directly thresholding the
inspection score — roughly 0–13 points is an A, 14–27 is a B, 28+ is a C (the mean score by grade above
shows exactly that separation). `grade_ord` isn't independent information about the restaurant; it's a
coarser, bucketed *version* of the score, computed by the same agency from the same inspection. Feeding
it into a model that predicts score is close to feeding the model a rounded-off copy of the answer.

This is **target leakage**: a feature derived from (or nearly equivalent to) the target, so a model built
with it looks far more accurate than it actually is. It's one of the most common ways a model quietly
cheats without a single line of buggy code. The fix is always the same question: *could I have known
this before the outcome I'm predicting, and was it created independently of that outcome?* `is_critical`
passes that test. `grade_ord` does not.

Compare this to 311's `complaint_grouped`: both caused a big jump in R². One held up under that question.
The other didn't. Same jump, opposite explanations — which is why the jump alone was never enough to
trust.

### Checking the model: residual plots

Same three checks as Part 4: centered on 0, no curve, no funnel.

In [ ]:
residuals = y_test - __________

plt.scatter(y_pred, residuals, alpha=0.4, s=15)
plt.axhline(0, color="#9AA5B1", linestyle="--")
plt.xlabel("Predicted score")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residuals vs. Predicted Score")
plt.show()

`is_critical` and `grade_ord` only take 2 and 3 distinct values, so this model can only make a handful of
distinct predictions — expect the dots to cluster into six thin vertical bands. Each band still has a
fair amount of spread (residuals of roughly ±40 points), but no obvious curve or funnel.

**This plot cannot tell you about leakage.** It only checks for a pattern the model missed — it has no way
to know that one of your features was built from your target. A model can pass this check with flying
colors and still be leaking. Leakage lives upstream, in how the features were built; you have to go
looking for it directly, the way the ablation above did.

### Two datasets, two different reasons to double-check

311 gave you a big jump in R² that was genuine — different complaint types really do get handled on
different timelines. The restaurant data gave you a big jump that was an illusion — `grade_ord` looks
predictive only because it's built from the target. The same tool caught both: don't trust a coefficient
or an R² at face value, fit the pieces separately, and ask why a feature works before you trust that it
does.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect07_regression_exercise.ipynb`.

---

## Part 6 — Cheat Sheet

| Task | Code |
|---|---|
| Fit a linear regression | `LinearRegression().fit(X_train, y_train)` |
| Predict | `model.predict(X_test)` |
| Coefficients / intercept | `model.coef_`, `model.intercept_` |
| Bundle preprocessing + model | `Pipeline([("preprocessor", ct), ("regressor", LinearRegression())])` |
| MAE | `mean_absolute_error(y_test, y_pred)` |
| MSE / RMSE | `mean_squared_error(y_test, y_pred)`, then `np.sqrt(...)` |
| R² | `r2_score(y_test, y_pred)` |
| Residuals | `y_test - y_pred` |
| Ablation (which feature matters?) | fit on one feature at a time, compare R² |
| *(Optional)* Compress a right-skewed target | `np.log1p(y)`, reverse with `np.expm1(...)` |

---

## Part 7 — Key Terms

- **Regression**: predicting a numeric target from one or more features.
- **$X$, $y$, $\hat{y}$**: $X$ is the input feature(s); $y$ is the actual target; $\hat{y}$ ("y-hat") is
  the model's predicted target.
- **Simple vs. multiple linear regression**: one feature vs. several features, same underlying idea.
- **Residual**: the difference between an actual value and the model's prediction, $y_i - \hat{y}_i$.
- **Least squares**: choosing model parameters that minimize the residual sum of squares (RSS).
- **RSS (residual sum of squares)**: $\sum (y_i - \hat{y}_i)^2$ — the quantity linear regression
  minimizes.
- *(Optional)* **Normal equation**: the closed-form matrix solution for least-squares coefficients,
  $\beta = (X^TX)^{-1}X^Ty$ — scikit-learn uses this idea internally; you won't need to compute it by
  hand in this course.
- **MAE / MSE / RMSE**: average error size, in original units (MAE, RMSE) or squared units (MSE); RMSE
  and MSE penalize large errors more than MAE does.
- **R² (coefficient of determination)**: the fraction of the target's variance the model explains,
  relative to always predicting the mean. R² = 0.80 does not mean 80% accuracy.
- **Residual plot**: a plot of residuals against predicted values. Check for three things: centered on
  0, no curve, no funnel.
- **Target leakage**: when a feature is derived from (or nearly equivalent to) the target, so a model
  built with it looks far more accurate than it genuinely is. Ask: could I have known this feature
  before the outcome, and was it created independently of it?
- *(Optional)* **Right-skewed target**: a target with a long tail of unusually large values; mean well
  above median is a quick sign of it. A log-transform (`np.log1p`) is a common fix.